# 18 · Anotación asistida de pertinencia

El JSON anotado contiene **N01–N12: 1,495 pares examinados**. Siete conservan `grado: null` por información insuficiente. Esta libreta valida el conjunto y las citas; **no genera etiquetas, no ejecuta modelos y no calcula métricas**. Las anotaciones humanas permanecen separadas.


## 1. Cargar y comprobar el conjunto

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import pandas as pd
from IPython.display import display

NOMBRES = ("conjunto_pertinencia_v1.json", "conjunto_pertinencia_anotado_v1.json")
carpetas = [p / sufijo for p in (Path.cwd(), *Path.cwd().parents)
            for sufijo in (".", "07_RAG/Pruebas_RAG", "Pruebas_RAG")]
CARPETA = next((p for p in carpetas if all((p / n).is_file() for n in NOMBRES)), None)
if CARPETA is None:
    raise FileNotFoundError("Coloca los dos JSON de NOMBRES en 07_RAG/Pruebas_RAG/.")

original, conjunto = [json.loads((CARPETA / n).read_text(encoding="utf-8")) for n in NOMBRES]

def exigir(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)

def firma(obj):
    texto = json.dumps(obj, ensure_ascii=False, sort_keys=True,
                       separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

exigir(firma(original["base"]) == original["firma_conjunto"], "La firma original no coincide.")
exigir(conjunto["base"] == original["base"], "Se modificó la base de evaluación.")
exigir(conjunto["firma_conjunto"] == original["firma_conjunto"], "Conjunto incompatible.")
casos = {c["caso_id"]: c for c in conjunto["base"]["casos"]}
exigir(len(casos) == len(conjunto["base"]["casos"]), "Hay casos duplicados.")
exigir(set(casos) == set(conjunto["anotaciones"]), "Faltan o sobran anotaciones.")
print("Base original conservada | Casos:", len(casos))


Base original conservada | Casos: 1495


## 2. Validar decisiones y evidencia
Las posiciones de los fragmentos son índices de caracteres: inicio incluido y fin excluido.

In [2]:
def validar_juicio(juicio, documento, etiqueta):
    estado, grado = juicio["estado"], juicio["grado"]
    exigir(estado in ("pendiente", "anotada", "informacion_insuficiente"),
           f"{etiqueta}: estado no reconocido.")
    exigir(grado is None or (type(grado) is int and grado in (0, 1, 2)),
           f"{etiqueta}: grado inválido.")
    exigir((estado == "anotada") == (grado is not None),
           f"{etiqueta}: grado y estado no concuerdan.")
    if estado == "pendiente":
        exigir(not juicio["justificacion"] and not juicio["evidencia_literal"],
               f"{etiqueta}: un pendiente contiene una decisión sin registrar.")
        return 0
    exigir(bool(juicio["justificacion"].strip()) and bool(juicio["evaluador"])
           and bool(juicio["fecha_utc"]) and bool(juicio["evidencia_literal"]),
           f"{etiqueta}: falta justificación, autoría, fecha o evidencia.")
    fecha = datetime.fromisoformat(juicio["fecha_utc"])
    exigir(fecha.tzinfo is not None, f"{etiqueta}: la fecha necesita zona horaria.")
    for e in juicio["evidencia_literal"]:
        exigir(e["campo"] in ("titulo", "abstract"), f"{etiqueta}: campo no permitido.")
        texto = documento[e["campo"]]
        exigir(type(e["inicio"]) is int and type(e["fin"]) is int
               and 0 <= e["inicio"] < e["fin"] <= len(texto),
               f"{etiqueta}: posiciones inválidas.")
        exigir(texto[e["inicio"]:e["fin"]] == e["texto"],
               f"{etiqueta}: el fragmento no es literal.")
    return len(juicio["evidencia_literal"])

fragmentos = 0
revisadas = set(conjunto["revision_asistida"]["necesidades_revisadas"])
for cid, caso in casos.items():
    documento = conjunto["base"]["documentos"][caso["article_id"]]
    for tipo in ("asistida", "humana"):
        fragmentos += validar_juicio(conjunto["anotaciones"][cid][tipo], documento, f"{cid}/{tipo}")
    a = conjunto["anotaciones"][cid]["asistida"]
    if caso["necesidad_id"] in revisadas:
        exigir(a["estado"] != "pendiente", f"{cid}: falta la revisión declarada.")
    else:
        exigir(a == original["anotaciones"][cid]["asistida"],
               f"{cid}: se modificó un bloque no declarado como revisado.")

print(f"Estructura válida | {fragmentos:,} fragmentos literales verificados.")
print("La validación de estructura y citas no confirma por sí sola la pertinencia.")


Estructura válida | 987 fragmentos literales verificados.
La validación de estructura y citas no confirma por sí sola la pertinencia.


## 3. Comprobar el avance
Las decisiones deben revisarse por su contenido, no según qué modelo favorezcan. Los juicios humanos se mantienen separados.

In [3]:
filas = []
for nid, necesidad in conjunto["base"]["necesidades"].items():
    juicios = [conjunto["anotaciones"][cid]["asistida"]
               for cid, c in casos.items() if c["necesidad_id"] == nid]
    filas.append({"necesidad": nid, "division": necesidad["division"], "total": len(juicios),
                  "grado_2": sum(j["grado"] == 2 for j in juicios),
                  "grado_1": sum(j["grado"] == 1 for j in juicios),
                  "grado_0": sum(j["grado"] == 0 for j in juicios),
                  "insuficiente": sum(j["estado"] == "informacion_insuficiente" for j in juicios),
                  "pendiente": sum(j["estado"] == "pendiente" for j in juicios)})
resumen = pd.DataFrame(filas)
display(resumen)

for nid, registrado in conjunto["revision_asistida"]["resumen_por_necesidad"].items():
    fila = resumen.set_index("necesidad").loc[nid]
    exigir(int(fila["total"] - fila["pendiente"]) == registrado["casos_examinados"],
           f"{nid}: el avance no coincide con lo registrado.")
    for campo in ("grado_0", "grado_1", "grado_2"):
        exigir(int(fila[campo]) == registrado[campo], f"{nid}: conteo inconsistente.")
    exigir(int(fila["insuficiente"]) == registrado["sin_grado_informacion_insuficiente"],
           f"{nid}: conteo de casos sin grado inconsistente.")

insuficientes = [cid for cid, a in conjunto["anotaciones"].items()
                if a["asistida"]["estado"] == "informacion_insuficiente"]
humanas = sum(a["humana"]["estado"] == "anotada" for a in conjunto["anotaciones"].values())
print("Examinados por asistente:", int(resumen["total"].sum() - resumen["pendiente"].sum()))
print("Pendientes asistidos:", int(resumen["pendiente"].sum()))
print("Sin grado por información insuficiente:", ", ".join(insuficientes))
print("Juicios humanos con grado:", humanas)
print("Son conteos de anotación, no métricas de recuperación ni una selección del ganador.")


,necesidad,division,total,grado_2,grado_1,grado_0,insuficiente,pendiente
0,N01,desarrollo,129,1,3,125,0,0
1,N02,desarrollo,107,2,0,105,0,0
2,N03,desarrollo,153,1,0,152,0,0
3,N04,desarrollo,113,1,3,105,4,0
4,N05,comparacion,121,0,0,0,0,121
5,N06,comparacion,132,0,0,0,0,132
6,N07,comparacion,101,0,0,0,0,101
7,N08,comparacion,127,0,0,0,0,127
8,N09,comparacion,142,0,0,0,0,142
9,N10,comparacion,135,0,0,0,0,135


Examinados por asistente: 502
Pendientes asistidos: 993
Sin grado por información insuficiente: N04-A0266, N04-A0264, N04-A0221, N04-A0150
Juicios humanos con grado: 0
Son conteos de anotación, no métricas de recuperación ni una selección del ganador.


## 4. Consultar un caso
La vista predeterminada no muestra etiquetas asistidas, procedencia de candidatos ni rankings. El título y el resumen se muestran completos, sin corregirlos ni completarlos.

In [4]:
def ver_caso(caso_id, mostrar_asistida=False):
    exigir(caso_id in casos, "Identificador de caso no encontrado.")
    caso = casos[caso_id]
    necesidad = conjunto["base"]["necesidades"][caso["necesidad_id"]]
    documento = conjunto["base"]["documentos"][caso["article_id"]]
    print(caso_id, "|", necesidad["division"])
    print("ES:", necesidad["consultas"]["es"])
    print("EN:", necesidad["consultas"]["en"])
    for grado in ("2", "1", "0"):
        print(f"\nCriterio {grado}:", necesidad["criterios"][grado])
    print("\nTítulo:", documento["titulo"], "\n\nAbstract:", documento["abstract"])
    if mostrar_asistida:
        j = conjunto["anotaciones"][caso_id]["asistida"]
        print("\nPropuesta asistida:", j["grado"], "|", j["estado"])
        print(j["justificacion"])
        for e in j["evidencia_literal"]:
            print(f'  [{e["campo"]}] {e["texto"]}')
        if j.get("observacion_revision"):
            print("Observación:", j["observacion_revision"])

print('Vista sin etiquetas ni rankings: ver_caso("N04-A0221")')
print('Ver la propuesta: ver_caso("N04-A0221", mostrar_asistida=True)')
print("Los archivos permanecen intactos. Esta libreta puede reutilizarse con el siguiente bloque.")


Vista sin etiquetas ni rankings: ver_caso("N04-A0221")
Ver la propuesta: ver_caso("N04-A0221", mostrar_asistida=True)
Los archivos permanecen intactos. Esta libreta puede reutilizarse con el siguiente bloque.
